<a href="https://colab.research.google.com/github/LMMinier/nueronce/blob/main/cfna_colab_training_FIXED_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CFNA: corpus → parameters (Colab training journal)

Turns the open-source corpus stack into trained CFNA weights, then runs the
conversational SFT stage and lets you chat with the result — with automatic checkpoint fallback and resumability. The corpus is persisted in Google Drive and restored automatically. After a Colab runtime
restart, rerun setup cell 1 and Drive cell 3, then resume at base-training cell 6.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

Honest framing (see `PLAN.md` / `docs/BREAKTHROUGH_MAP.md`): SFT *shapes*
competence but cannot *create* it. The gate is **base held-out bpb < 1.8**
before SFT; the 35M-rung acceptance bars are listed at the bottom.


In [1]:
# 1) Setup: clone the repo, pin the branch, and make cfna importable.
#    Safe to rerun from any working directory after a Colab restart.
from pathlib import Path
import importlib.util
import os
import sys

REPO = "LMMinier/nueronce"
BRANCH = "claude/cfna-neural-core-verify-ldvgn3"  # change only when intentionally switching branches
REPO_DIR = Path("/content/nueronce")

if not REPO_DIR.exists():
    !git clone --branch "$BRANCH" "https://github.com/$REPO" "$REPO_DIR"

%cd /content/nueronce
!git fetch origin "$BRANCH"
!git checkout "$BRANCH"
!git pull --ff-only origin "$BRANCH"
%pip install -q numpy datasets pytest cryptography cffi

# Running `python scripts/foo.py` makes Python search /content/nueronce/scripts
# first, not the repository root. Export the root explicitly for every child
# process and add it to this notebook kernel as well.
REPO_ROOT = str(REPO_DIR.resolve())
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

spec = importlib.util.find_spec("cfna")
if spec is None:
    raise ModuleNotFoundError(
        f"cfna is still not importable from {REPO_ROOT}. "
        "Check that /content/nueronce/cfna exists and that the branch checkout succeeded."
    )

import torch
import cfna
print("repo:", REPO_ROOT)
print("cfna:", getattr(cfna, "__file__", None) or spec.origin)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


/content/nueronce
From https://github.com/LMMinier/nueronce
 * branch            claude/cfna-neural-core-verify-ldvgn3 -> FETCH_HEAD
Already on 'claude/cfna-neural-core-verify-ldvgn3'
Your branch is up to date with 'origin/claude/cfna-neural-core-verify-ldvgn3'.
From https://github.com/LMMinier/nueronce
 * branch            claude/cfna-neural-core-verify-ldvgn3 -> FETCH_HEAD
Already up to date.
repo: /content/nueronce
cfna: /content/nueronce/cfna/__init__.py
torch 2.11.0+cu128 | cuda: True Tesla T4


In [2]:
# 2) Safety gate: the fp16/AMP numerics tests must be green before any AMP run
%cd /content/nueronce
!PYTHONPATH="/content/nueronce${PYTHONPATH:+:$PYTHONPATH}" \
  python -m pytest tests/test_gpu_amp.py tests/test_prompting.py tests/test_chat_format_drift.py -q


/content/nueronce
..............                                                           [100%]


In [3]:
# 3) (Recommended) Mount Google Drive so checkpoints survive disconnects
USE_DRIVE = True
CKPT_ROOT = "checkpoints"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_ROOT = "/content/drive/MyDrive/cfna_checkpoints"
    os.makedirs(CKPT_ROOT, exist_ok=True)
print("checkpoints ->", CKPT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoints -> /content/drive/MyDrive/cfna_checkpoints


In [4]:
# 4) BUILD OR RESTORE CORPUS — crash-safe, streamed, local-first
from pathlib import Path
from dataclasses import asdict
import json
import os
import shutil
import subprocess
import sys
import textwrap

REPO_DIR = Path("/content/nueronce")
LOCAL_CORPUS = Path("/content/cfna_corpus_stack")
PARTS_DIR = Path("/content/cfna_corpus_parts")
DRIVE_CORPUS = Path("/content/drive/MyDrive/cfna_corpus_stack")
DRIVE_TEMP = Path("/content/drive/MyDrive/cfna_corpus_stack.incomplete")
HF_CACHE = Path("/content/huggingface_cache")

os.chdir(REPO_DIR)

os.environ["PYTHONPATH"] = (
    str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HF_CACHE.mkdir(parents=True, exist_ok=True)


def verify_corpus(corpus_dir: Path):
    manifest_path = corpus_dir / "manifest.jsonl"

    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing manifest: {manifest_path}")

    records = [
        json.loads(line)
        for line in manifest_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if not records:
        raise RuntimeError(f"Empty manifest: {manifest_path}")

    missing = []
    empty = []

    for record in records:
        path = corpus_dir / record["path"]

        if not path.is_file():
            missing.append(str(path))
        elif path.stat().st_size == 0:
            empty.append(str(path))

    if missing:
        raise FileNotFoundError(
            "Manifest references missing files:\n" +
            "\n".join(missing[:20])
        )

    if empty:
        raise RuntimeError(
            "Corpus contains empty files:\n" +
            "\n".join(empty[:20])
        )

    splits = {record.get("split") for record in records}

    if "train" not in splits or "val" not in splits:
        raise RuntimeError(
            f"Corpus requires train and val splits; found {sorted(splits)}"
        )

    total_bytes = sum(
        (corpus_dir / record["path"]).stat().st_size
        for record in records
    )

    return records, total_bytes


# ------------------------------------------------------------
# First restore a valid corpus from Drive when one already exists
# ------------------------------------------------------------

drive_valid = False

try:
    drive_records, drive_bytes = verify_corpus(DRIVE_CORPUS)
    drive_valid = True

    print(
        f"Valid persistent corpus found: "
        f"{len(drive_records)} files / {drive_bytes / 1e6:.2f} MB"
    )
except Exception as exc:
    print("No valid persistent corpus found:", exc)


if drive_valid:
    local_valid = False

    try:
        local_records, local_bytes = verify_corpus(LOCAL_CORPUS)
        local_valid = local_bytes == drive_bytes
    except Exception:
        local_valid = False

    if not local_valid:
        print("Restoring corpus from Google Drive to local storage...")

        if LOCAL_CORPUS.exists():
            shutil.rmtree(LOCAL_CORPUS)

        shutil.copytree(DRIVE_CORPUS, LOCAL_CORPUS)

    records, total_bytes = verify_corpus(LOCAL_CORPUS)

    CORPUS_DIR = str(LOCAL_CORPUS)

    print("Corpus ready:", CORPUS_DIR)
    print("Manifest records:", len(records))
    print("Corpus size:", f"{total_bytes / 1e6:.2f} MB")

else:
    # ------------------------------------------------------------
    # Build each source in its own process.
    # Streaming=True prevents full datasets loading into RAM.
    # ------------------------------------------------------------

    worker_path = Path("/content/cfna_build_one_source.py")

    worker_code = r'''
from __future__ import annotations

import argparse
import json
import re
from datetime import date
from hashlib import sha256
from pathlib import Path

from datasets import load_dataset

from cfna.corpus.stack import get_entry

END_DOCUMENT = "\n\n<|END_DOCUMENT|>\n\n"
SPACE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = "\n".join(
        SPACE.sub(" ", line).strip()
        for line in text.split("\n")
    )
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


def extract_text(entry, row):
    if entry.document_template == "dolly":
        instruction = str(row.get("instruction") or "").strip()
        context = str(row.get("context") or "").strip()
        response = str(row.get("response") or "").strip()

        if not instruction or not response:
            return None

        parts = [f"Instruction: {instruction}"]

        if context:
            parts.append(f"Context: {context}")

        parts.append(f"Response: {response}")
        return "\n".join(parts)

    if entry.document_template == "oasst1":
        if row.get("lang") and row.get("lang") != "en":
            return None

        text = str(row.get("text") or "").strip()
        role = row.get("role") or "message"

        return f"{role}: {text}" if text else None

    for field in entry.text_fields:
        value = row.get(field)

        if isinstance(value, str) and value.strip():
            return value

    return None


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source", required=True)
    parser.add_argument("--out", required=True)
    parser.add_argument("--target-bytes", type=int, required=True)
    parser.add_argument("--val-every", type=int, default=20)
    parser.add_argument("--min-chars", type=int, default=200)
    args = parser.parse_args()

    entry = get_entry(args.source)
    out_dir = Path(args.out)
    text_dir = out_dir / "stack_text"

    text_dir.mkdir(parents=True, exist_ok=True)

    train_path = text_dir / f"{entry.source_id}__train.txt"
    val_path = text_dir / f"{entry.source_id}__val.txt"

    load_args = {
        "path": entry.dataset_name,
        "split": entry.split,
        # Force streaming even for registry entries marked non-streaming.
        # This avoids loading an entire dataset into Colab memory.
        "streaming": True,
    }

    if entry.dataset_config:
        load_args["name"] = entry.dataset_config

    print("Loading:", entry.dataset_name)
    print("Config:", entry.dataset_config)
    print("Streaming:", True)

    dataset = load_dataset(**load_args)

    total_content_bytes = 0
    split_docs = {"train": 0, "val": 0}
    split_content_bytes = {"train": 0, "val": 0}

    handles = {
        "train": train_path.open("w", encoding="utf-8"),
        "val": val_path.open("w", encoding="utf-8"),
    }

    try:
        for row in dataset:
            text = extract_text(entry, row)

            if not text:
                continue

            text = clean_text(text)

            if len(text) < args.min_chars:
                continue

            encoded = text.encode("utf-8")

            if total_content_bytes + len(encoded) > args.target_bytes:
                break

            document_number = (
                split_docs["train"] +
                split_docs["val"] +
                1
            )

            split = (
                "val"
                if args.val_every > 0
                and document_number % args.val_every == 0
                else "train"
            )

            handles[split].write(text)
            handles[split].write(END_DOCUMENT)

            split_docs[split] += 1
            split_content_bytes[split] += len(encoded)
            total_content_bytes += len(encoded)

            if document_number % 5000 == 0:
                print(
                    f"{entry.source_id}: "
                    f"{document_number:,} docs / "
                    f"{total_content_bytes / 1e6:.2f} MB",
                    flush=True,
                )

    finally:
        for handle in handles.values():
            handle.close()

    records = []

    for split, path in (
        ("train", train_path),
        ("val", val_path),
    ):
        if split_docs[split] == 0:
            path.unlink(missing_ok=True)
            continue

        digest = sha256(path.read_bytes()).hexdigest()

        records.append({
            "document_id": f"{entry.source_id}_{split}",
            "title": f"{entry.name} ({split})",
            "author": "dataset",
            "document_type": entry.role,
            "source_collection": entry.name,
            "source_locator": entry.dataset_page,
            "files_page": entry.files_page,
            "license": entry.license,
            "license_id": entry.license,
            "commercial_use": (
                "noncommercial" not in entry.license.lower()
            ),
            "attribution_required": any(
                token in entry.license.lower()
                for token in ("by", "sharing", "share")
            ),
            "language": "en",
            "publication_year": None,
            "retrieved_at": date.today().isoformat(),
            "content_hash": f"sha256:{digest}",
            "quality_score": 1.0,
            "n_bytes": split_content_bytes[split],
            "n_docs": split_docs[split],
            "split": split,
            "bucket": f"phase_{entry.phase}_{entry.role}",
            "phase": entry.phase,
            "role": entry.role,
            "path": str(path.relative_to(out_dir)),
        })

    manifest = out_dir / "manifest.jsonl"

    with manifest.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record) + "\n")

    if not records:
        raise RuntimeError(
            f"{entry.source_id} produced no valid documents"
        )

    print(
        f"COMPLETE {entry.source_id}: "
        f"{sum(split_docs.values()):,} documents / "
        f"{total_content_bytes / 1e6:.2f} MB"
    )


if __name__ == "__main__":
    main()
'''

    worker_path.write_text(
        textwrap.dedent(worker_code),
        encoding="utf-8",
    )

    # Per-source byte budgets.
    # The total requested content is approximately 400 MB.
    SOURCE_JOBS = [
        ("tinystories", 140_000_000),
        ("cosmopedia_100k", 120_000_000),
        ("fineweb_edu_sample_10bt", 90_000_000),
        ("open_web_math", 50_000_000),
    ]

    if PARTS_DIR.exists():
        shutil.rmtree(PARTS_DIR)

    PARTS_DIR.mkdir(parents=True, exist_ok=True)

    successful_parts = []
    failures = []

    for source_id, byte_budget in SOURCE_JOBS:
        part_dir = PARTS_DIR / source_id

        if part_dir.exists():
            shutil.rmtree(part_dir)

        print("\n" + "=" * 70)
        print(
            f"Building {source_id} "
            f"with target {byte_budget / 1e6:.0f} MB"
        )
        print("=" * 70)

        command = [
            sys.executable,
            "-u",
            str(worker_path),
            "--source",
            source_id,
            "--out",
            str(part_dir),
            "--target-bytes",
            str(byte_budget),
            "--val-every",
            "20",
        ]

        result = subprocess.run(
            command,
            cwd=REPO_DIR,
            env=os.environ.copy(),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )

        print(result.stdout)

        if result.returncode == 0:
            try:
                part_records, part_bytes = verify_corpus(part_dir)

                successful_parts.append(part_dir)

                print(
                    f"Accepted {source_id}: "
                    f"{len(part_records)} manifest records / "
                    f"{part_bytes / 1e6:.2f} MB"
                )

            except Exception as exc:
                failures.append(
                    (source_id, f"verification failed: {exc}")
                )
        else:
            failures.append(
                (
                    source_id,
                    f"worker exited with code {result.returncode}",
                )
            )

            print(
                f"WARNING: {source_id} failed, "
                "continuing with the remaining sources."
            )

    if not successful_parts:
        raise RuntimeError(
            "Every corpus source failed.\n"
            "Failures:\n" +
            "\n".join(
                f"- {source}: {reason}"
                for source, reason in failures
            )
        )

    # ------------------------------------------------------------
    # Merge completed source directories
    # ------------------------------------------------------------

    if LOCAL_CORPUS.exists():
        shutil.rmtree(LOCAL_CORPUS)

    combined_text_dir = LOCAL_CORPUS / "stack_text"
    combined_text_dir.mkdir(parents=True, exist_ok=True)

    combined_records = []

    for part_dir in successful_parts:
        part_manifest = part_dir / "manifest.jsonl"

        part_records = [
            json.loads(line)
            for line in part_manifest.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        for record in part_records:
            source_path = part_dir / record["path"]
            destination_path = LOCAL_CORPUS / record["path"]

            destination_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(source_path, destination_path)
            combined_records.append(record)

    manifest_path = LOCAL_CORPUS / "manifest.jsonl"

    with manifest_path.open("w", encoding="utf-8") as handle:
        for record in combined_records:
            handle.write(json.dumps(record) + "\n")

    # Save the full source registry/catalog.
    from cfna.corpus.stack import CORPUS_STACK

    catalog = []

    for entry in CORPUS_STACK:
        item = asdict(entry)
        item["text_fields"] = list(entry.text_fields)
        catalog.append(item)

    (LOCAL_CORPUS / "corpus_stack_catalog.json").write_text(
        json.dumps(catalog, indent=2),
        encoding="utf-8",
    )

    records, total_bytes = verify_corpus(LOCAL_CORPUS)

    if total_bytes < 50_000_000:
        raise RuntimeError(
            f"Only {total_bytes / 1e6:.2f} MB was built. "
            "That is too small to continue base training safely."
        )

    print("\nCombined corpus verified.")
    print("Manifest records:", len(records))
    print("Combined size:", f"{total_bytes / 1e6:.2f} MB")

    if failures:
        print("\nSources that failed:")

        for source, reason in failures:
            print(f" - {source}: {reason}")

    # ------------------------------------------------------------
    # Persist only after the local corpus is fully verified
    # ------------------------------------------------------------

    print("\nSaving completed corpus to Google Drive...")

    if DRIVE_TEMP.exists():
        shutil.rmtree(DRIVE_TEMP)

    shutil.copytree(LOCAL_CORPUS, DRIVE_TEMP)

    temp_records, temp_bytes = verify_corpus(DRIVE_TEMP)

    if temp_bytes != total_bytes:
        raise RuntimeError(
            "Drive copy size does not match the local corpus."
        )

    if DRIVE_CORPUS.exists():
        shutil.rmtree(DRIVE_CORPUS)

    DRIVE_TEMP.rename(DRIVE_CORPUS)

    drive_records, drive_bytes = verify_corpus(DRIVE_CORPUS)

    CORPUS_DIR = str(LOCAL_CORPUS)

    print("\nCORPUS BUILD COMPLETE")
    print("Local corpus:", CORPUS_DIR)
    print("Drive corpus:", DRIVE_CORPUS)
    print("Manifest:", LOCAL_CORPUS / "manifest.jsonl")
    print("Manifest records:", len(drive_records))
    print("Corpus size:", f"{drive_bytes / 1e6:.2f} MB")

Valid persistent corpus found: 2 files / 408.89 MB
Corpus ready: /content/cfna_corpus_stack
Manifest records: 2
Corpus size: 408.89 MB


In [5]:
# 5) OPTIONAL: publish the corpus on a dedicated GitHub branch.
#    OFF by default. This stages a COPY before splitting large files, so the
#    training corpus and its manifest are never damaged.
PUSH_CORPUS = False

if PUSH_CORPUS:
    from getpass import getpass
    from pathlib import Path
    import os
    import shutil

    REPO_DIR = Path("/content/nueronce")
    SOURCE_CORPUS = Path(CORPUS_DIR)
    STAGING_CORPUS = REPO_DIR / "corpus_stack_publish"

    if STAGING_CORPUS.exists():
        shutil.rmtree(STAGING_CORPUS)
    shutil.copytree(SOURCE_CORPUS, STAGING_CORPUS)

    token = getpass("GitHub token (repo write): ")
    os.chdir(REPO_DIR)

    # Split only the disposable publishing copy.
    !find corpus_stack_publish -name '*.txt' -size +90M \
        -exec sh -c 'split -b 90m "$1" "$1.part-" && rm "$1"' _ {} \;

    !git config user.email "luisminier79@gmail.com"
    !git config user.name "LMMinier"
    !git checkout -B corpus-stack
    !git add -f corpus_stack_publish
    !git commit -m "Add downloaded corpus stack publishing copy" || echo "nothing to commit"
    !git push -u https://$token@github.com/$REPO corpus-stack
    !git checkout $BRANCH

    del token


In [ ]:
# 6) BASE PRETRAIN — repaired resume, full error logging, safe retry

from pathlib import Path
from datetime import datetime
import gc
import json
import os
import shutil
import subprocess
import sys

import torch

REPO_DIR = Path("/content/nueronce")
LOCAL_CORPUS = Path("/content/cfna_corpus_stack")
DRIVE_CORPUS = Path("/content/drive/MyDrive/cfna_corpus_stack")

if "CKPT_ROOT" not in globals():
    raise RuntimeError("Run Cell 3 first to mount Google Drive.")

if not (REPO_DIR / "cfna").is_dir():
    raise FileNotFoundError(
        "/content/nueronce/cfna is missing. Run Cell 1 first."
    )

TRAIN_SCRIPT = REPO_DIR / "scripts/train_checkpoint.py"

if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing trainer: {TRAIN_SCRIPT}")

os.chdir(REPO_DIR)

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(REPO_DIR)
    + os.pathsep
    + env.get("PYTHONPATH", "")
)
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"


# ------------------------------------------------------------
# Verify corpus
# ------------------------------------------------------------

def verify_corpus(corpus_dir: Path):
    corpus_dir = Path(corpus_dir)
    manifest = corpus_dir / "manifest.jsonl"

    if not manifest.is_file():
        raise FileNotFoundError(f"Missing manifest: {manifest}")

    records = []

    for line_number, line in enumerate(
        manifest.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise RuntimeError(
                f"Bad JSON in manifest line {line_number}: {exc}"
            ) from exc

        if "path" not in record or "split" not in record:
            raise RuntimeError(
                f"Manifest line {line_number} lacks path or split."
            )

        records.append(record)

    if not records:
        raise RuntimeError(f"Manifest is empty: {manifest}")

    missing = []
    empty = []
    actual_bytes = 0

    for record in records:
        file_path = corpus_dir / record["path"]

        if not file_path.is_file():
            missing.append(str(file_path))
            continue

        size = file_path.stat().st_size

        if size == 0:
            empty.append(str(file_path))
            continue

        actual_bytes += size

    if missing:
        raise FileNotFoundError(
            "Corpus files are missing:\n" + "\n".join(missing[:20])
        )

    if empty:
        raise RuntimeError(
            "Corpus files are empty:\n" + "\n".join(empty[:20])
        )

    splits = {str(record["split"]) for record in records}

    if not {"train", "val"}.issubset(splits):
        raise RuntimeError(
            f"Corpus requires train and val splits; found {sorted(splits)}"
        )

    if actual_bytes < 1_000_000:
        raise RuntimeError(
            f"Corpus is only {actual_bytes / 1e6:.2f} MB."
        )

    return records, actual_bytes


# Prefer Cell 4's CORPUS_DIR, then local cache, then Drive.
candidates = []

if "CORPUS_DIR" in globals():
    candidates.append(Path(CORPUS_DIR))

candidates.extend([LOCAL_CORPUS, DRIVE_CORPUS])

selected = None
records = None
corpus_bytes = None
errors = []

for candidate in candidates:
    try:
        found_records, found_bytes = verify_corpus(candidate)
        selected = candidate
        records = found_records
        corpus_bytes = found_bytes
        break
    except Exception as exc:
        errors.append(f"{candidate}: {exc}")

if selected is None:
    raise RuntimeError(
        "No valid corpus was found.\n\n"
        + "\n".join(errors)
        + "\n\nRun the corrected Cell 4 first."
    )

# Training reads faster from /content than directly from Drive.
if selected == DRIVE_CORPUS:
    print("Restoring corpus from Drive to local Colab storage...")

    if LOCAL_CORPUS.exists():
        shutil.rmtree(LOCAL_CORPUS)

    shutil.copytree(DRIVE_CORPUS, LOCAL_CORPUS)

    records, corpus_bytes = verify_corpus(LOCAL_CORPUS)
    selected = LOCAL_CORPUS

CORPUS_DIR = str(selected)


# ------------------------------------------------------------
# Patch trainer's resumed optimizer state onto CUDA
# ------------------------------------------------------------

trainer_text = TRAIN_SCRIPT.read_text(encoding="utf-8")

patch_marker = "# CFNA_RESUME_OPTIMIZER_DEVICE_FIX"

if patch_marker not in trainer_text:
    original = """    model.to(device)
    valb = [b.to(device) for b in valb]
"""

    replacement = """    model.to(device)

    # CFNA_RESUME_OPTIMIZER_DEVICE_FIX
    # torch.load(..., map_location="cpu") leaves Adam's exp_avg and
    # exp_avg_sq buffers on CPU. Move them to the model device before
    # the first optimizer step.
    for state in opt.state.values():
        for key, value in list(state.items()):
            if torch.is_tensor(value):
                state[key] = value.to(device)

    valb = [b.to(device) for b in valb]
"""

    if original not in trainer_text:
        raise RuntimeError(
            "Could not patch train_checkpoint.py because its expected "
            "model.to(device) section has changed."
        )

    trainer_text = trainer_text.replace(original, replacement, 1)
    TRAIN_SCRIPT.write_text(trainer_text, encoding="utf-8")

    print("Patched resumed optimizer tensors for CUDA.")
else:
    print("Trainer resume patch already installed.")


# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Select Runtime → Change runtime type → T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

try:
    free_bytes, total_bytes = torch.cuda.mem_get_info()

    print(
        "GPU memory:",
        f"{free_bytes / 1e9:.2f} GB free / "
        f"{total_bytes / 1e9:.2f} GB total",
    )
except Exception:
    pass


# ------------------------------------------------------------
# Validate existing checkpoint
# ------------------------------------------------------------

checkpoint_root = Path(CKPT_ROOT)
checkpoint_root.mkdir(parents=True, exist_ok=True)

BASE_CKPT = checkpoint_root / "cfna_base_35m.pt"
LOG_PATH = checkpoint_root / "cfna_base_35m_last_run.log"

from cfna.model import CONFIG_PRESETS

expected_config = vars(CONFIG_PRESETS["base_35m"]())
resume_allowed = False


def preserve_bad_checkpoint(path: Path, reason: str):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = path.with_name(
        f"{path.name}.{reason}_{timestamp}"
    )

    shutil.move(str(path), str(backup))

    print("Preserved unusable checkpoint as:")
    print(backup)


if BASE_CKPT.exists():
    print("Inspecting existing checkpoint:", BASE_CKPT)
    print("Checkpoint size:", f"{BASE_CKPT.stat().st_size / 1e6:.2f} MB")

    try:
        checkpoint = torch.load(
            BASE_CKPT,
            map_location="cpu",
            weights_only=False,
        )

        required = {"state_dict", "config"}

        if not isinstance(checkpoint, dict):
            raise TypeError("Checkpoint root is not a dictionary.")

        missing_keys = required - set(checkpoint)

        if missing_keys:
            raise KeyError(
                f"Checkpoint lacks keys: {sorted(missing_keys)}"
            )

        if checkpoint["config"] != expected_config:
            preserve_bad_checkpoint(BASE_CKPT, "wrong_config")
        else:
            resume_allowed = True

            print(
                "Valid resume checkpoint:",
                f"step {checkpoint.get('step', 0)}",
            )

        del checkpoint
        gc.collect()

    except Exception as exc:
        print("Checkpoint validation failed:", repr(exc))

        if BASE_CKPT.exists():
            preserve_bad_checkpoint(BASE_CKPT, "corrupt")

else:
    print("No existing checkpoint. Starting a new base model.")


# ------------------------------------------------------------
# Run and show complete trainer output
# ------------------------------------------------------------

def run_training(batch_size: int):
    command = [
        sys.executable,
        "-u",
        str(TRAIN_SCRIPT),
        "--preset", "base_35m",
        "--corpus", CORPUS_DIR,
        "--minutes", "170",
        "--seq", "192",
        "--batch", str(batch_size),
        "--lr", "3e-4",
        "--amp",
        "--device", "cuda",
        "--out", str(BASE_CKPT),
    ]

    if resume_allowed:
        command.append("--resume")

    print("\nLaunching:")
    print(" ".join(command))
    print()

    output_lines = []

    with LOG_PATH.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()
            output_lines.append(line)

        return_code = process.wait()

    return return_code, "".join(output_lines)


print("\nRepository:", REPO_DIR)
print("Corpus:", CORPUS_DIR)
print("Corpus records:", len(records))
print("Corpus size:", f"{corpus_bytes / 1e6:.2f} MB")
print("Checkpoint:", BASE_CKPT)
print("Log:", LOG_PATH)

# base_35m is intended to run at batch 16 with AMP.
# Retry smaller only when the actual error is CUDA OOM.
for batch_size in [16, 8, 4]:
    return_code, output = run_training(batch_size)

    if return_code == 0:
        print("\nBASE TRAINING COMPLETED")
        print("Checkpoint:", BASE_CKPT)
        break

    output_lower = output.lower()

    is_oom = (
        "cuda out of memory" in output_lower
        or "outofmemoryerror" in output_lower
        or "cuda error: out of memory" in output_lower
    )

    if is_oom and batch_size != 4:
        print(
            f"\nCUDA memory error at batch {batch_size}. "
            "Retrying with a smaller batch..."
        )

        gc.collect()
        torch.cuda.empty_cache()
        continue

    tail = "\n".join(output.splitlines()[-60:])

    raise RuntimeError(
        "Base training failed.\n\n"
        "Actual trainer output:\n"
        "----------------------------------------\n"
        f"{tail}\n"
        "----------------------------------------\n"
        f"Complete log: {LOG_PATH}"
    )
else:
    raise RuntimeError(
        "Training failed even with batch size 4. "
        f"Read the complete log at {LOG_PATH}"
    )

Patched resumed optimizer tensors for CUDA.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 15.53 GB free / 15.64 GB total
Inspecting existing checkpoint: /content/drive/MyDrive/cfna_checkpoints/cfna_base_35m.pt
Checkpoint size: 350.57 MB
Checkpoint validation failed: RuntimeError('PytorchStreamReader failed locating file data/694: file not found. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.')
Preserved unusable checkpoint as:
/content/drive/MyDrive/cfna_checkpoints/cfna_base_35m.pt.corrupt_20260702_142532

Repository: /content/nueronce
Corpus: /content/cfna_corpus_stack
Corpus records: 2
Corpus size: 408.89 MB
Checkpoint: /content/drive/MyDrive/cfna_checkpoints/cfna_base_35m.pt
Log: /content/drive/MyDrive/cfna_checkpoints/cfna_base_35m_last_run.log

Launchi

In [ ]:
# 7) Build the mixed conversational SFT dataset (54K+ records, canonical
#    format, <=25% per register, leakage-checked). Deterministic; ~10 s.
%cd /content/nueronce
!PYTHONPATH="/content/nueronce${PYTHONPATH:+:$PYTHONPATH}" \
  python scripts/build_conversation_sft.py --out-dir data/conversation_sft


In [ ]:
# 8) CONVERSATIONAL SFT: response-masked loss on the canonical format.
#    First run warm-starts from the base checkpoint; later runs auto-resume.
#    The checkpoint resolver below also handles short runs that produced only
#    latest.pt before the first periodic "best" validation.
from pathlib import Path
import subprocess
import sys
import os
import torch

SFT_DIR = Path(CKPT_ROOT) / "conv_35m"
SFT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CKPT = Path(BASE_CKPT)
LATEST_CKPT = SFT_DIR / "latest.pt"
BEST_CKPT = SFT_DIR / "best.pt"

if LATEST_CKPT.exists():
    start_args = ["--resume"]
    print("Resuming conversational SFT from:", LATEST_CKPT)
else:
    if not BASE_CKPT.exists():
        raise FileNotFoundError(
            f"Base checkpoint not found: {BASE_CKPT}. Run cell 6 first."
        )
    start_args = ["--init-from", str(BASE_CKPT)]
    print("Warm-starting conversational SFT from:", BASE_CKPT)

cmd = [
    sys.executable, "scripts/train_conversation.py",
    "--data", "data/conversation_sft",
    "--preset", "base_35m",
    *start_args,
    "--loss", "response",
    "--out-dir", str(SFT_DIR),
    "--minutes", "60",
    "--batch", "16",
    "--lr", "1e-4",
    "--val-every", "100",
    "--amp",
]
run_env = os.environ.copy()
run_env["PYTHONPATH"] = "/content/nueronce" + os.pathsep + run_env.get("PYTHONPATH", "")
subprocess.run(cmd, check=True, cwd="/content/nueronce", env=run_env)

# train_conversation.py writes best.pt only after a periodic validation. A short
# run can legitimately finish with latest.pt only, so make that final checkpoint
# loadable by the evaluation/chat cells instead of crashing.
if not BEST_CKPT.exists():
    if not LATEST_CKPT.exists():
        raise FileNotFoundError(
            f"Training finished without a checkpoint in {SFT_DIR}. "
            "Inspect the training output above for the real failure."
        )

    checkpoint = torch.load(LATEST_CKPT, map_location="cpu", weights_only=False)
    checkpoint.setdefault("meta", {})
    checkpoint["meta"]["checkpoint_selection"] = "latest_fallback"
    checkpoint["meta"]["best_checkpoint_missing"] = True

    temp_path = BEST_CKPT.with_suffix(".pt.tmp")
    torch.save(checkpoint, temp_path)
    temp_path.replace(BEST_CKPT)
    print("No periodic best.pt was created; promoted latest.pt to:", BEST_CKPT)
else:
    print("Best checkpoint ready:", BEST_CKPT)


In [ ]:
# 9) Evaluate the exact checkpoint selected by the training cell.
from pathlib import Path
import subprocess
import sys
import os

SFT_DIR = Path(CKPT_ROOT) / "conv_35m"
BEST_CKPT = SFT_DIR / "best.pt"
LATEST_CKPT = SFT_DIR / "latest.pt"
EVAL_CKPT = BEST_CKPT if BEST_CKPT.exists() else LATEST_CKPT

if not EVAL_CKPT.exists():
    raise FileNotFoundError(
        f"No conversational checkpoint found in {SFT_DIR}. Run cell 8 first."
    )

print("Evaluating:", EVAL_CKPT)
run_env = os.environ.copy()
run_env["PYTHONPATH"] = "/content/nueronce" + os.pathsep + run_env.get("PYTHONPATH", "")
subprocess.run([
    sys.executable,
    "scripts/eval_inference_phase2.py",
    "--write-suite",
    "--checkpoint", str(EVAL_CKPT),
], check=True, cwd="/content/nueronce", env=run_env)


In [ ]:
# 10) CHAT with the model (canonical format stamped in checkpoint metadata)
from pathlib import Path
import sys

REPO_ROOT = "/content/nueronce"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from cfna.chat import Conversation, load_checkpoint

SFT_DIR = Path(CKPT_ROOT) / "conv_35m"
checkpoint_candidates = [
    SFT_DIR / "best.pt",
    SFT_DIR / "latest.pt",
]
CHAT_CKPT = next((path for path in checkpoint_candidates if path.exists()), None)

if CHAT_CKPT is None:
    discovered = sorted(
        Path(CKPT_ROOT).rglob("*.pt"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    found = "\n".join(f" - {path}" for path in discovered[:20]) or " - none"
    raise FileNotFoundError(
        f"No conversational checkpoint found in {SFT_DIR}. Run cell 8 first.\n"
        f"Checkpoints currently visible under {CKPT_ROOT}:\n{found}"
    )

print("Loading:", CHAT_CKPT)
model, ckpt = load_checkpoint(str(CHAT_CKPT))
meta = ckpt.get("meta", {})
print(
    "prompt_format:", meta.get("prompt_format"),
    "| val acc:", meta.get("best_val_acc", "not recorded"),
    "| selection:", meta.get("checkpoint_selection", "best-by-validation"),
)

convo = Conversation(model, system="You are CFNA.", temperature=0.0)
for question in [
    "Hello",
    "What is two plus two?",
    "What is the capital of France?",
    "Is 8 even or odd?",
    "Thank you",
]:
    print(f"User: {question}\nCFNA: {convo.say(question)}\n")


In [ ]:
# 11) Push training metrics + curves back to the repo (never the raw weights;
#     those live in Drive). The cloud session reads these for go/no-go.
from getpass import getpass
token = getpass("GitHub token (repo write): ")
!git add metrics/ && git commit -m "Add Colab training metrics" || echo 'nothing to commit'
!git push https://$token@github.com/$REPO $BRANCH
del token

## Gates and acceptance (from `PLAN.md` / `docs/CODEX_HANDOFF.md`)

1. **Do not SFT until base held-out bpb < 1.8** — keep re-running cell 6.
2. 35M-rung acceptance: bpb ≤ 1.5 · choice-ranking beats chance by ≥15 pts on
   ≥3 MCQ subjects · ≥60% valid non-echo stop-terminated answers · 5/5
   grammatical transcripts. Only then move to `--preset base_90m`.
3. Never eval a checkpoint in a prompt format it wasn't trained on
   (`docs/FORMAT.md`).
4. Report weak results in `docs/reports/` — they are part of the record.